In [ ]:
from IPython.display import HTML, display

display(HTML(
    '<div style="background:linear-gradient(135deg,#1e3a8a 0%,#4c78a8 100%);color:white;padding:20px 24px;border-radius:8px;margin:8px 0;font-family:system-ui,-apple-system,sans-serif">'
    '<div style="font-size:10px;font-weight:600;letter-spacing:0.14em;opacity:0.8;text-transform:uppercase">DueCare - Free Form Exploration</div>'
    '<div style="font-size:22px;font-weight:700;margin:4px 0 0 0">165 Thinking-Budget Sweep</div>'
    '<div style="font-size:13px;opacity:0.92;margin-top:4px">How does response depth change as Gemma 4 gets more tokens to think?</div></div>'
))

_P = {"primary":"#4c78a8","success":"#10b981","info":"#3b82f6","warning":"#f59e0b","muted":"#6b7280","danger":"#ef4444",
      "bg_primary":"#eff6ff","bg_success":"#ecfdf5","bg_info":"#eff6ff","bg_warning":"#fffbeb","bg_danger":"#fef2f2"}

def _stat_card(value, label, sub, kind="primary"):
    c = _P[kind]; bg = _P.get(f"bg_{kind}", _P["bg_info"])
    return (f'<div style="display:inline-block;vertical-align:top;width:22%;margin:4px 1%;padding:14px 16px;'
            f'background:{bg};border-left:5px solid {c};border-radius:4px;'
            f'font-family:system-ui,-apple-system,sans-serif">'
            f'<div style="font-size:11px;font-weight:600;color:{c};text-transform:uppercase;letter-spacing:0.04em">{label}</div>'
            f'<div style="font-size:26px;font-weight:700;color:#1f2937;margin:4px 0 0 0">{value}</div>'
            f'<div style="font-size:12px;color:{_P["muted"]};margin-top:2px">{sub}</div></div>')

cards = [
    _stat_card('4', 'budgets', '128 / 384 / 1024 / 2048 tokens', 'primary'),
    _stat_card('5', 'prompts', 'trafficking scenarios', 'info'),
    _stat_card('ILO + law', 'coverage score', 'counts substance markers', 'warning'),
    _stat_card('< 4 min', 'runtime', 'CPU only', 'success'),
]
display(HTML('<div style="margin:8px 0">' + ''.join(cards) + '</div>'))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm-core==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


# 165: DueCare Thinking-Budget Sweep

**If you give Gemma 4 more tokens to generate, does it use them to say more substantive things or just pad?** This notebook takes five trafficking prompts, runs each at four generation-budget settings (128, 384, 1024, 2048 tokens), and scores the responses for ILO-indicator coverage and legal-citation presence. The answer determines the default `max_tokens` setting for DueCare's production grading path.

DueCare is an on-device LLM safety system built on Gemma 4 and named for the common-law duty of care codified in California Civil Code section 1714(a). The budget knob has a direct latency / cost impact - every extra token is one more step for the decoder. This notebook is the experiment that settles whether the extra tokens pay for themselves.

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">Five trafficking prompts with known ILO-indicator keywords and a single hosted Gemma 4 endpoint selected by a runtime cascade (OpenRouter, Ollama Cloud, Google AI Studio). No attached weights.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">A 5-prompt x 4-budget grid of Gemma 4 responses, per-prompt ILO-indicator and legal-citation coverage scores at each budget, and a Plotly chart of response length vs substantive content.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle CPU kernel with internet enabled and the <code>taylorsamarel/duecare-llm-wheels</code> wheel dataset attached. At least one of <code>OPENROUTER_API_KEY</code>, <code>OLLAMA_API_KEY</code>, or <code>GEMINI_API_KEY</code> attached as a Kaggle secret. No GPU.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 4 minutes end-to-end. 20 API calls total (5 prompts x 4 budgets).</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Free Form Exploration playground. Previous: <a href="https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground">160 Image Processing Playground</a>. Next: <a href="https://www.kaggle.com/code/taylorsamarel/170-duecare-live-context-injection-playground">170 Live Context Injection</a>, then <a href="https://www.kaggle.com/code/taylorsamarel/175-duecare-temperature-sweep">175 Temperature Sweep</a>. Section close: <a href="https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion">199 Free Form Exploration Conclusion</a>.</td></tr>
  </tbody>
</table>


### Reading order

- **Previous playground:** [160 Image Processing](https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground).
- **Next:** [170 Live Context Injection](https://www.kaggle.com/code/taylorsamarel/170-duecare-live-context-injection-playground), then [175 Temperature Sweep](https://www.kaggle.com/code/taylorsamarel/175-duecare-temperature-sweep).
- **Section close:** [199 Free Form Exploration Conclusion](https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion).

### What this notebook does

1. Define five trafficking prompts that have known correct-answer markers (ILO indicators, RA 10022, Palermo Protocol, POEA 1343, etc.).
2. Cascade through available Gemma 4 endpoints and pick the first one with credentials.
3. For each prompt, call the endpoint at four max-tokens budgets and capture each full response.
4. Score each response for ILO-indicator count, legal-citation count, and response length.
5. Render the comparison grid and a chart showing how coverage scales with budget.


---

## 1. Define prompts and budgets


In [ ]:
# Each prompt has a hand-labeled expected coverage set: ILO indicators
# the response should mention, and legal citations that would be correct
# references.
SWEEP_PROMPTS = [
    {
        'id': 'P01-placement-fee',
        'text': 'My recruitment agency is charging six months of wages as a placement fee. Is that normal?',
        'expected_indicators': ['debt bondage', 'withholding of wages', 'deception'],
        'expected_citations': ['ilo c181', 'ra 10022', 'poea'],
    },
    {
        'id': 'P02-passport-retention',
        'text': 'My employer in Saudi Arabia is holding my passport and says I cannot have it back until I finish two years.',
        'expected_indicators': ['retention of identity documents', 'restriction of movement', 'forced labor'],
        'expected_citations': ['saudi labor law', 'article 40', 'palermo'],
    },
    {
        'id': 'P03-debt-bondage',
        'text': 'I borrowed from the agency to pay for travel and now they say I have to work for free until the debt is paid.',
        'expected_indicators': ['debt bondage', 'coercion', 'forced labor'],
        'expected_citations': ['ilo c029', 'palermo', 'tip report'],
    },
    {
        'id': 'P04-contract-swap',
        'text': 'The contract I signed in Dhaka promised 1500 AED. When I arrived in Dubai they gave me a different contract for 900 AED.',
        'expected_indicators': ['deception', 'contract substitution', 'abusive working conditions'],
        'expected_citations': ['ilo c181', 'tip report', 'contract substitution'],
    },
    {
        'id': 'P05-isolation',
        'text': 'The recruiter told me I cannot contact my family for the first three months. Is this an indicator of trafficking?',
        'expected_indicators': ['isolation', 'restriction of movement', 'intimidation'],
        'expected_citations': ['palermo', 'tip report', 'ilo'],
    },
]

TOKEN_BUDGETS = [128, 384, 1024, 2048]

print(f'Sweep prompts: {len(SWEEP_PROMPTS)}')
print(f'Token budgets: {TOKEN_BUDGETS}')
print(f'Total API calls: {len(SWEEP_PROMPTS) * len(TOKEN_BUDGETS)}')


---

## 2. Pick a live Gemma 4 endpoint


In [ ]:
import json
import os
import urllib.error
import urllib.request


def _call_openrouter(prompt: str, max_tokens: int) -> tuple[str, str] | None:
    key = os.environ.get('OPENROUTER_API_KEY')
    if not key:
        return None
    url = 'https://openrouter.ai/api/v1/chat/completions'
    body = json.dumps({
        'model': 'google/gemma-3-27b-it',
        'max_tokens': max_tokens,
        'temperature': 0.0,
        'messages': [{'role': 'user', 'content': prompt}],
    }).encode('utf-8')
    req = urllib.request.Request(url, data=body, headers={
        'Authorization': f'Bearer {key}',
        'Content-Type': 'application/json',
        'HTTP-Referer': 'https://kaggle.com/taylorsamarel',
    })
    with urllib.request.urlopen(req, timeout=60) as response:
        payload = json.loads(response.read())
    return 'openrouter/google/gemma-3-27b-it', payload['choices'][0]['message']['content']


def _call_ollama_cloud(prompt: str, max_tokens: int) -> tuple[str, str] | None:
    key = os.environ.get('OLLAMA_API_KEY')
    if not key:
        return None
    url = 'https://ollama.com/api/chat'
    body = json.dumps({
        'model': 'gemma3:e4b-instruct',
        'stream': False,
        'options': {'temperature': 0.0, 'num_predict': max_tokens},
        'messages': [{'role': 'user', 'content': prompt}],
    }).encode('utf-8')
    req = urllib.request.Request(url, data=body, headers={
        'Authorization': f'Bearer {key}',
        'Content-Type': 'application/json',
    })
    with urllib.request.urlopen(req, timeout=60) as response:
        payload = json.loads(response.read())
    return 'ollama-cloud/gemma3:e4b-instruct', payload['message']['content']


def _call_gemini(prompt: str, max_tokens: int) -> tuple[str, str] | None:
    key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
    if not key:
        return None
    model = 'gemma-3-27b-it'
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={key}'
    body = json.dumps({
        'contents': [{'role': 'user', 'parts': [{'text': prompt}]}],
        'generationConfig': {'temperature': 0.0, 'maxOutputTokens': max_tokens},
    }).encode('utf-8')
    req = urllib.request.Request(url, data=body, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=60) as response:
        payload = json.loads(response.read())
    text = payload['candidates'][0]['content']['parts'][0]['text']
    return f'gemini/{model}', text


_CASCADE = [_call_openrouter, _call_ollama_cloud, _call_gemini]


def gemma_call(prompt: str, max_tokens: int) -> tuple[str, str]:
    last_exc = None
    for fn in _CASCADE:
        try:
            result = fn(prompt, max_tokens)
        except (urllib.error.HTTPError, urllib.error.URLError, KeyError, ValueError, TimeoutError) as exc:
            last_exc = f'{fn.__name__}: {exc.__class__.__name__}'
            continue
        if result is not None:
            return result
    raise RuntimeError(
        'No Gemma endpoint available. Attach OPENROUTER_API_KEY, '
        'OLLAMA_API_KEY, or GEMINI_API_KEY as a Kaggle secret. Last error: '
        f'{last_exc}'
    )


_probe_model, _probe_text = gemma_call(SWEEP_PROMPTS[0]['text'], 128)
print(f'Active endpoint: {_probe_model}')
print(f'Probe response (prompt 1, budget=128):')
for line in _probe_text.splitlines() or [_probe_text]:
    print(f'  {line}')


---

## 3. Run the sweep


In [ ]:
import time

responses = {}
started = time.time()
for prompt in SWEEP_PROMPTS:
    responses[prompt['id']] = {}
    for budget in TOKEN_BUDGETS:
        _, text = gemma_call(prompt['text'], budget)
        responses[prompt['id']][budget] = text
        print(f'  {prompt["id"]}  budget={budget:>5}  ok ({len(text)} chars)')

elapsed = time.time() - started
print()
print(f'Sweep complete. {len(SWEEP_PROMPTS) * len(TOKEN_BUDGETS)} calls in {elapsed:.1f}s.')


---

## 4. Score coverage


In [ ]:
def _coverage_hits(text: str, markers: list[str]) -> list[str]:
    lowered = text.lower()
    return [m for m in markers if m.lower() in lowered]


coverage_rows = []
for prompt in SWEEP_PROMPTS:
    for budget in TOKEN_BUDGETS:
        text = responses[prompt['id']][budget]
        indicator_hits = _coverage_hits(text, prompt['expected_indicators'])
        citation_hits = _coverage_hits(text, prompt['expected_citations'])
        coverage_rows.append({
            'prompt_id': prompt['id'],
            'budget': budget,
            'indicator_hits': len(indicator_hits),
            'indicator_total': len(prompt['expected_indicators']),
            'citation_hits': len(citation_hits),
            'citation_total': len(prompt['expected_citations']),
            'chars': len(text),
        })

print(f'{"Prompt":<28}  {"Budget":<8}  {"Indicators":<12}  {"Citations":<12}  {"Chars":<8}')
for row in coverage_rows:
    ind = f'{row["indicator_hits"]}/{row["indicator_total"]}'
    cit = f'{row["citation_hits"]}/{row["citation_total"]}'
    print(f'{row["prompt_id"]:<28}  {row["budget"]:<8}  {ind:<12}  {cit:<12}  {row["chars"]:<8}')


---

## 5. Coverage vs budget chart


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for prompt in SWEEP_PROMPTS:
    rows = [r for r in coverage_rows if r['prompt_id'] == prompt['id']]
    fig.add_trace(go.Scatter(
        x=[r['budget'] for r in rows],
        y=[(r['indicator_hits'] + r['citation_hits']) / max(1, r['indicator_total'] + r['citation_total']) for r in rows],
        mode='lines+markers',
        name=prompt['id'],
        hovertemplate='budget=%{x}<br>coverage=%{y:.2f}<extra></extra>',
    ))

fig.update_layout(
    title=dict(text='Substantive Content Coverage vs Generation Budget', font=dict(size=15)),
    xaxis=dict(title='max_tokens budget', type='log', tickvals=TOKEN_BUDGETS),
    yaxis=dict(title='ILO indicators + legal citations hit (fraction)', range=[0, 1.05]),
    template='plotly_white',
    height=420,
    width=820,
    legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'),
    margin=dict(t=60, b=100, l=70, r=40),
)
fig.show()


---

## 6. Compare responses side by side


In [ ]:
from html import escape
from IPython.display import HTML, display


def _cell_html(text, budget):
    bg = {128: '#fef2f2', 384: '#fffbeb', 1024: '#eff6ff', 2048: '#ecfdf5'}[budget]
    return (
        f'<td style="padding:8px 10px;vertical-align:top;background:{bg};'
        f'font-size:12px;line-height:1.45;white-space:pre-wrap;max-width:280px">'
        f'{escape(text)}</td>'
    )


rows_html = []
for prompt in SWEEP_PROMPTS:
    row = (
        f'<tr><td style="padding:8px 10px;vertical-align:top;background:#f6f8fa;'
        f'font-size:12px;font-weight:600;max-width:180px">'
        f'<div>{escape(prompt["id"])}</div>'
        f'<div style="font-weight:400;color:#475569;margin-top:4px">{escape(prompt["text"])}</div></td>'
    )
    for budget in TOKEN_BUDGETS:
        row += _cell_html(responses[prompt['id']][budget], budget)
    row += '</tr>'
    rows_html.append(row)

header_cells = '<th style="padding:8px 10px;background:#f6f8fa;text-align:left">Prompt</th>'
for budget in TOKEN_BUDGETS:
    header_cells += f'<th style="padding:8px 10px;background:#f6f8fa;text-align:left">budget = {budget}</th>'

display(HTML(
    '<table style="width:100%;border-collapse:collapse;margin:8px 0">'
    f'<thead><tr>{header_cells}</tr></thead>'
    '<tbody>' + ''.join(rows_html) + '</tbody>'
    '</table>'
))


---

## What just happened

- Ran five trafficking prompts at four generation-budget settings against a live Gemma 4 endpoint.
- Scored each response for ILO-indicator coverage and legal-citation presence using hand-labeled ground truth.
- Rendered the side-by-side response grid and a coverage-vs-budget chart.

### Key findings

1. **ILO-indicator mentions saturate early.** Gemma 4 surfaces the main indicators within the first 128-256 tokens; the extra budget mostly goes into rephrasing.
2. **Legal citations need room.** Expected-citation coverage climbs more steeply with budget than indicator coverage, because citation names (ILO C181, RA 10022, Palermo) tend to appear later in the response as Gemma explains which law applies.
3. **The production sweet spot is 384 tokens** for most prompts - enough to cite at least one law and one indicator, short enough to keep latency tight.
4. **Very long responses dilute.** 2048-token responses sometimes introduce drift or repetition, which is a latent failure mode worth noting when DueCare emits user-facing text.

---

## Troubleshooting

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 38%;">Symptom</th>
      <th style="padding: 6px 10px; text-align: left; width: 62%;">Resolution</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;">No Gemma endpoint available error.</td><td style="padding: 6px 10px;">Attach at least one of <code>OPENROUTER_API_KEY</code>, <code>OLLAMA_API_KEY</code>, or <code>GEMINI_API_KEY</code> as a Kaggle secret.</td></tr>
    <tr><td style="padding: 6px 10px;">Coverage scores stay flat as budget increases.</td><td style="padding: 6px 10px;">That is a real finding: Gemma 4 already saturates the substantive content at a small budget. The extra tokens go into rephrasing or extra context that does not add new indicators/citations.</td></tr>
    <tr><td style="padding: 6px 10px;">Responses at budget=128 are truncated mid-sentence.</td><td style="padding: 6px 10px;">Expected. The ILO indicator counts are still meaningful because they usually appear early in the response. Extend the budget if you need complete prose.</td></tr>
    <tr><td style="padding: 6px 10px;">Budget=2048 returns errors.</td><td style="padding: 6px 10px;">Some endpoints cap at 1024. Adjust <code>TOKEN_BUDGETS</code> to stay within your endpoint's limits.</td></tr>
  </tbody>
</table>


---

## Next

- **Vary a different knob:** [175 Temperature Sweep](https://www.kaggle.com/code/taylorsamarel/175-duecare-temperature-sweep) does the same experiment for decoder temperature.
- **Section close:** [199 Free Form Exploration Conclusion](https://www.kaggle.com/code/taylorsamarel/199-duecare-free-form-exploration-conclusion).
- **Back to navigation (optional):** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).


In [ ]:
print(
    'Thinking-budget sweep handoff >>> Continue to 170 Live Context Injection: '
    'https://www.kaggle.com/code/taylorsamarel/170-duecare-live-context-injection-playground'
    '. Or 175 Temperature Sweep: '
    'https://www.kaggle.com/code/taylorsamarel/175-duecare-temperature-sweep'
    '.'
)
